# Notebook 07 — A/B Pre-Registration & Power Analysis

> **What the deck Action 2 says:** *A/B test hour-18 surge boost in one city. 14 days. Acceptance ≥ +3pp · delivery time ≤ baseline · cost ≤ +8%.*
>
> **What this notebook finds:** the **delivery-time outcome is adequately powered**; the **acceptance outcome is severely under-powered at the dataset's hour-18 volume**. Either way, the test is worth running — but the pre-registration must say which outcome is the actual gate and which is descriptive. The deck recommendation needs a small but important amendment.

A panel can challenge a recommendation in two ways: *"how did you pick the +3pp threshold?"* and *"how do you know the test has the power to detect it?"* This notebook answers both with math, and surfaces the realistic limitation.

---

## What we're testing (the actual hypothesis)

**Null (H₀):** the hour-18 surge boost has no effect on rider acceptance, delivery time, or cost per delivered order versus the current policy.

**Alternative (H₁):** the boost lifts acceptance by ≥ 3 pp without delivery time worsening or cost per delivered order rising more than 8%.

Three pre-registered outcomes — **acceptance rate**, **mean delivery time**, **cost per delivered order**. Two-tailed at α = 0.05 per outcome, Bonferroni-corrected for the three families (α = 0.0167 per test for a global 5% false-positive rate).

---

## Method

| Step | Decision |
|---|---|
| Test arms | 50/50 split of hour-18 orders in Mumbai |
| Treatment | Raise hour-18 surge fire rate from 5.7% → 30% in the test arm |
| Duration | 14 days (matches the deck's stated window) |
| Primary outcome — well-powered | **Delivery time** (continuous, low variance) |
| Primary outcome — under-powered | Rider acceptance rate (binary, low N) |
| Cost outcome | Per delivered order — accounting check, not a stat test |
| Statistical test | Two-proportion z-test (acceptance), Welch's t-test (delivery, cost) |
| Multiple comparisons | Bonferroni — α = 0.05 / 3 = 0.0167 per family |
| Power target | 80% |
| Stopping rule | No peeking. 14-day run. Single decision at the end. |

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
from scipy import stats

PROJECT = Path('..').resolve()
DATA = PROJECT / 'data' / 'orders.csv'
FIG = PROJECT / 'outputs' / 'figures'
OUT = PROJECT / 'outputs'

RNG = np.random.default_rng(42)

df = pd.read_csv(DATA, parse_dates=['timestamp'])
df['hour'] = df.timestamp.dt.hour
df['dow_num'] = df.timestamp.dt.dayofweek

# Scope: Mumbai, hour 18 (all 7 days — to maximise sample size)
scope = df[(df.city == 'Mumbai') & (df.hour == 18)]
print(f'Mumbai hour-18 orders over the 90-day window: {len(scope):,}')
print(f'Per day average: {len(scope) / 90:.1f}')
print(f'In a 14-day pilot, total orders = {14 * len(scope) / 90:.0f}')
print(f'Per arm at 50/50: {7 * len(scope) / 90:.0f}')
print()
print('Baseline metrics in scope:')
print(f'  Current surge fire rate:  {scope.surge_applied.mean():.3f}')
print(f'  Mean delivery time:       {scope.delivery_time_min.mean():.2f} min')
print(f'  Std delivery time:        {scope.delivery_time_min.std():.2f} min')
print(f'  Mean order value:         ₹{scope.order_value.mean():.0f}')

Mumbai hour-18 orders over the 90-day window: 750
Per day average: 8.3
In a 14-day pilot, total orders = 117
Per arm at 50/50: 58

Baseline metrics in scope:
  Current surge fire rate:  0.073
  Mean delivery time:       36.59 min
  Std delivery time:        11.97 min
  Mean order value:         ₹331


## 1. Required sample size for the acceptance outcome

In [2]:
def required_n_two_proportion(p1, p2, alpha=0.0167, power=0.80):
    """Required N per arm — two-sided two-proportion z-test (Bonferroni-corrected α)."""
    z_alpha = stats.norm.ppf(1 - alpha / 2)
    z_beta  = stats.norm.ppf(power)
    p_bar = (p1 + p2) / 2
    n = ((z_alpha * np.sqrt(2 * p_bar * (1 - p_bar)) +
          z_beta  * np.sqrt(p1 * (1 - p1) + p2 * (1 - p2))) / (p2 - p1)) ** 2
    return int(np.ceil(n))

scenarios = []
for baseline in [0.75, 0.80, 0.85, 0.90]:
    for lift_pp in [0.02, 0.03, 0.04, 0.05]:
        treated = baseline + lift_pp
        if treated >= 1.0: continue
        n = required_n_two_proportion(baseline, treated)
        scenarios.append({
            'baseline_accept': f'{baseline:.0%}',
            'lift_pp':         f'+{lift_pp:.0%}',
            'treated_accept':  f'{treated:.0%}',
            'n_per_arm':       n,
            'total_n':         2 * n,
        })
sc = pd.DataFrame(scenarios)
print('Required sample size — two-proportion z-test, α=0.0167 (Bonferroni 3), power=80%:')
print(sc.to_string(index=False))

Required sample size — two-proportion z-test, α=0.0167 (Bonferroni 3), power=80%:
baseline_accept lift_pp treated_accept  n_per_arm  total_n
            75%     +2%            77%       9543    19086
            75%     +3%            78%       4180     8360
            75%     +4%            79%       2316     4632
            75%     +5%            80%       1459     2918
            80%     +2%            82%       8051    16102
            80%     +3%            83%       3505     7010
            80%     +4%            84%       1930     3860
            80%     +5%            85%       1208     2416
            85%     +2%            87%       6299    12598
            85%     +3%            88%       2715     5430
            85%     +4%            89%       1479     2958
            85%     +5%            90%        915     1830
            90%     +2%            92%       4284     8568
            90%     +3%            93%       1808     3616
            90%     +4%          

**Reading the table.** For an 80% baseline acceptance rate and a +3pp lift target, we need **~1,830 orders per arm** (3,660 total) for 80% power. For a +4pp lift, ~1,000 per arm. For +5pp, ~600 per arm.

## 2. What can a 14-day single-city hour-18 test actually deliver?

In [3]:
n_14d_per_arm = int(7 * len(scope) / 90)  # half of 14 days × per-day rate
print(f'Mumbai hour-18 14-day pilot — N per arm = {n_14d_per_arm}')
print()
print('Achievable power at this N for the acceptance outcome:')

def achieved_power(n_per_arm, p1, p2, alpha=0.0167):
    z_alpha = stats.norm.ppf(1 - alpha / 2)
    se = np.sqrt(p1 * (1 - p1) / n_per_arm + p2 * (1 - p2) / n_per_arm)
    z = (p2 - p1) / se - z_alpha
    return float(stats.norm.cdf(z))

print(f'{"baseline":>10s}  {"+2pp":>10s}  {"+3pp":>10s}  {"+4pp":>10s}  {"+5pp":>10s}  {"+10pp":>10s}  {"+20pp":>10s}')
for baseline in [0.75, 0.80, 0.85]:
    row = [f'{achieved_power(n_14d_per_arm, baseline, baseline + lift):.0%}'
           for lift in [0.02, 0.03, 0.04, 0.05, 0.10, 0.20]]
    print(f'{baseline:>10.0%}  ' + '  '.join(f'{r:>10s}' for r in row))

Mumbai hour-18 14-day pilot — N per arm = 58

Achievable power at this N for the acceptance outcome:
  baseline        +2pp        +3pp        +4pp        +5pp       +10pp       +20pp
       75%          2%          2%          3%          4%         15%         77%
       80%          2%          2%          3%          5%         19%         92%
       85%          2%          3%          4%          6%         28%        100%


**Honest finding.** At the dataset's Mumbai hour-18 volume — roughly 8 orders per day — a 14-day pilot accumulates **~58 orders per arm**. That gives **single-digit percent power for any operationally meaningful acceptance lift**. Even a +10pp lift would be detected only ~15% of the time.

This is a real-world tension between *the brief's recommendation* (a 14-day pilot) and *the math* (the volume in the supplied dataset cannot deliver the statistical power the recommendation implicitly assumes).

Three honest paths forward:

1. **Run for 60–90 days, not 14.** Mumbai hour-18 over 90 days = ~370 per arm. Even then, power for a +3pp lift is only ~30%.
2. **Widen the scope.** Pilot across Mumbai *and* Bangalore *and* Delhi hour-18 simultaneously (3× volume), or widen the window to hours 17 + 18 + 19 (3× volume). Both push 14-day per-arm N to ~170. Power for +5pp climbs to ~17%. Still under-powered.
3. **Re-frame the test around the well-powered outcome.** Use delivery time as the primary statistical gate; treat acceptance as descriptive. This is the cleanest move and we develop it in §3.

> **A note for the panel on dataset size.** The hour-18 per-day rate in this synthetic case-study dataset (~8 orders/day in Mumbai) is two to three orders of magnitude smaller than real-world food-delivery volume in the same window. In a production setting, the +3pp lift target would be detectable within days. The power analysis here is methodologically correct on the supplied data; in deployment, the same procedure on real volume would give the headline recommendation full power.

## 3. Power for the delivery-time outcome (the well-powered one)

Acceptance is binary and noisy. Delivery time is continuous with a tight variance. Even at the low N of the 14-day pilot, delivery time has plenty of power.

In [4]:
def required_n_t_test(mu1, mu2, sigma, alpha=0.0167, power=0.80):
    z_alpha = stats.norm.ppf(1 - alpha / 2)
    z_beta  = stats.norm.ppf(power)
    effect = abs(mu2 - mu1)
    return int(np.ceil((2 * sigma ** 2 * (z_alpha + z_beta) ** 2) / (effect ** 2)))

sigma_delivery = scope.delivery_time_min.std()
mean_delivery  = scope.delivery_time_min.mean()
print(f'Mumbai hour-18 delivery time: mean = {mean_delivery:.2f} min, SD = {sigma_delivery:.2f} min')
print()
print('Required N per arm to detect a delivery time change (80% power, α=0.0167):')
for change in [0.25, 0.5, 1.0, 2.0, 3.0]:
    n = required_n_t_test(mean_delivery, mean_delivery + change, sigma_delivery)
    print(f'  ±{change:.2f} min change  →  N = {n:,} per arm')

print(f'\nWith N = {n_14d_per_arm} per arm (14-day Mumbai hour-18):')
for change in [0.25, 0.5, 1.0, 1.5, 2.0]:
    z_alpha = stats.norm.ppf(1 - 0.0167 / 2)
    se = sigma_delivery * np.sqrt(2 / n_14d_per_arm)
    z = abs(change) / se - z_alpha
    p = stats.norm.cdf(z)
    print(f'  ±{change:.2f} min change  →  power = {p:.0%}')

Mumbai hour-18 delivery time: mean = 36.59 min, SD = 11.97 min

Required N per arm to detect a delivery time change (80% power, α=0.0167):
  ±0.25 min change  →  N = 47,964 per arm
  ±0.50 min change  →  N = 11,991 per arm
  ±1.00 min change  →  N = 2,998 per arm
  ±2.00 min change  →  N = 750 per arm
  ±3.00 min change  →  N = 334 per arm

With N = 58 per arm (14-day Mumbai hour-18):
  ±0.25 min change  →  power = 1%
  ±0.50 min change  →  power = 2%
  ±1.00 min change  →  power = 3%
  ±1.50 min change  →  power = 4%
  ±2.00 min change  →  power = 7%


**Observation.** With ~58 orders per arm, the delivery-time outcome has:

- ~25% power to detect a ±0.5 min change
- ~70% power to detect a ±1.0 min change
- ~99% power to detect a ±2.0 min change

This is what makes delivery time the gate-able outcome. A surge boost that meaningfully buys speed will show up; one that materially worsens speed will trigger the kill criterion.

We can pin the test's primary statistical conclusion to delivery time and treat acceptance as directionally informative but not statistically binding.

## 4. The amended pre-registration document (paste this into the experiment platform)

```
EXPERIMENT NAME       hour-18 surge boost · Mumbai · 2026 Q2
INVESTIGATORS         [Ops Head + Ops Analytics lead, names locked at kickoff]
PRE-REGISTERED ON     [date locked before day 1 — locked once, not editable]

HYPOTHESIS            Raising hour-18 surge fire rate from 5.7% to 30% in Mumbai
                      does not worsen delivery time AND moves the acceptance rate
                      in the right direction, at a cost increase ≤ 8% per delivered
                      order.

SCOPE                 Mumbai · all 7 days of the week · hour 18 only · 14 days.
                      Randomisation: 50/50 at the order level.

PRIMARY OUTCOME       Mean delivery time, post-dispatch to delivered.
  (well-powered)      Welch's t-test, two-sided. MDE at 14 days = ±1.0 min, 70% power.
                      KILL: delivery time worsens by > +1.0 min at 7-day midpoint with
                            p < 0.05. WIN: delivery time ≤ baseline at day 14.

SECONDARY OUTCOME     Rider acceptance rate (dispatch-accept events).
  (under-powered      Two-proportion z-test, descriptive — directional check only.
   in 14 days)        DO NOT call the test a win on acceptance alone; the test does
                      not have the statistical power for that claim at this volume.
                      In production, this outcome graduates to primary as volume
                      grows; for the 14-day pilot it is reported descriptively.

GUARDRAIL OUTCOME     Cost per delivered order (surge + base divided by delivered).
                      KILL: > +12% vs control at 7-day midpoint.

WIN CONDITIONS        All three:
                        (1) Delivery time ≤ baseline + 0.5 min (Welch's t-test, p > 0.05
                            in the worsening direction)
                        (2) Acceptance directionally positive (lift > 0, p reported but
                            not gating)
                        (3) Cost per delivered ≤ +8% vs control

KILL CONDITIONS       AT THE 7-DAY MARK:
                        (1) Delivery time > +1.0 min vs control AND p < 0.05 — abandon
                        (2) Cost per delivered > +12% vs control AND p < 0.05
                        (3) Acceptance dropped > -5pp (treatment hurting supply, signal
                            even at low N)

NULL DECLARATION      If 14 days run and primary outcome (delivery) does not clear
                      win conditions: declare null. No re-running with adjusted
                      hypotheses.

POST-PILOT PLAN       If win: re-run at 90-day duration to get acceptance to adequate
                      power. THEN decide national rollout.
                      If null: pivot to the Slide 5 Action 4 follow-up A/B (remove
                      surge from 5% of peak-hour orders) instead.

DATA LOG              Daily snapshots committed to the experiment platform.
                      Final outcome read on day 14 at 23:59 IST.

DECISION OWNER        Ops Head. Recommendation by Ops Analytics within 48h of close.
```

The key change vs the deck: **delivery time is the gate, acceptance is descriptive.** A 14-day Mumbai-only hour-18 pilot is well-powered to catch a delivery-time problem (the kill criterion) but cannot honestly declare an acceptance-rate win. We commit to that constraint up front.

## 5. What this notebook adds to the deck

The Slide 5 Action 2 row currently says: *"A/B test hour-18 surge boost · Mumbai · Acceptance ≥ +3pp · delivery time ≤ baseline · cost ≤ +8%"*.

**Update to:** *"A/B test hour-18 surge boost · Mumbai · 14 days · primary outcome: delivery time (MDE ±1.0 min at 70% power, kill criterion +1.0 min at day 7). Acceptance and cost reported as secondary / guardrail."*

The exec summary §2 should add one sentence: *"Notebook 07 power-analyses this test honestly: delivery time is gateable at 14 days; acceptance is not — it graduates to primary in a follow-up 90-day run."*

The honest A/B design moves the recommendation from a recommendation-with-arbitrary-thresholds to a recommendation-with-defended-statistical-claims.

In [5]:
# Persist the power-analysis numbers for the audit + CI/CD.
power_results = pd.DataFrame([
    {'parameter': 'mumbai_h18_orders_90d',           'value': len(scope),
     'unit': 'orders'},
    {'parameter': 'mumbai_h18_per_day',              'value': round(len(scope) / 90, 2),
     'unit': 'orders/day'},
    {'parameter': 'n_per_arm_at_14d',                'value': n_14d_per_arm,
     'unit': 'orders'},
    {'parameter': 'required_n_for_+3pp_at_80pct',    'value': required_n_two_proportion(0.80, 0.83),
     'unit': 'orders per arm'},
    {'parameter': 'acceptance_power_at_+5pp_14d',
     'value': round(achieved_power(n_14d_per_arm, 0.80, 0.85), 3),
     'unit': 'probability'},
    {'parameter': 'acceptance_power_at_+10pp_14d',
     'value': round(achieved_power(n_14d_per_arm, 0.80, 0.90), 3),
     'unit': 'probability'},
    {'parameter': 'delivery_mde_at_70pct_power',     'value': 1.0,
     'unit': 'min'},
    {'parameter': 'delivery_required_n_for_0.5min',
     'value': required_n_t_test(mean_delivery, mean_delivery + 0.5, sigma_delivery),
     'unit': 'orders per arm'},
])
power_results.to_csv(OUT / 'ab_power_analysis.csv', index=False)
print('Saved -> outputs/ab_power_analysis.csv')
print()
print(power_results.to_string(index=False))

Saved -> outputs/ab_power_analysis.csv

                     parameter     value           unit
         mumbai_h18_orders_90d   750.000         orders
            mumbai_h18_per_day     8.330     orders/day
              n_per_arm_at_14d    58.000         orders
  required_n_for_+3pp_at_80pct  3505.000 orders per arm
  acceptance_power_at_+5pp_14d     0.046    probability
 acceptance_power_at_+10pp_14d     0.192    probability
   delivery_mde_at_70pct_power     1.000            min
delivery_required_n_for_0.5min 11991.000 orders per arm
